# Kickstarter launch scouting: end-to-end notebook (Group 7, MLOps)

**Data source:** *Kickstarter Projects* on Kaggle (uploader: kemical), file `ks-projects-201801.csv`: 378,661 campaigns launched on Kickstarter between 2009 and early 2018. https://www.kaggle.com/datasets/kemical/kickstarter-projects (licence: check the dataset page before publishing). The older file in the same dataset, `ks-projects-201612.csv`, is not used: its column names carry trailing spaces and it has four empty overflow columns.

**Business question:** we are an early-stage investor scouting Kickstarter. On the day a campaign **launches**, can we tell from what is visible at that moment (category, country, goal, campaign length, timing, title) whether it will **reach its funding goal**? We then contact the most promising creators first, before the campaign has proven itself and before other investors notice. Funding success is a **proxy** for early demand, not a guarantee of investment returns.

**Logic of the notebook.** This is the "lab" of our MLOps project: it explores the data and fixes every choice the pipeline will use. The pipeline (`src/`) reproduces only the final choices; nothing is tuned there.

1. **EDA:** understand the campaigns and the quality of the data.
2. **Cleaning and validation rules:** fixed, stateless rules (`clean_data.py`) and checks (`validate.py`).
3. **Split:** train, validation and test, before anything is fitted (`main.py`).
4. **Model comparison:** logistic regression, decision tree and random forest on the same recipe (`features.py`) and the same split; tuning on the training rows only.
5. **Chosen model:** quality gate on validation (`evaluate.py`), drivers, stability over time, test split opened once.
6. **Business use:** the investor's shortlist and how much better it is than picking at random.
7. **Final choices:** one cell with every setting for `TEAM_BRIEF.md` and `config.yaml`.

**Rule we follow throughout:** only information known **at launch** is used as input, and nothing that learns from the data is fitted before the split or on rows other than the training rows.

# 1. Exploratory data analysis

## 1.1 Data and variable types

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)

RAW_DATA_PATH = Path('..') / 'data' / 'raw' / 'ks-projects-201801.csv'   # data is never committed to Git
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print('Campaigns:', df.shape[0], '| Variables:', df.shape[1])
df.head()

In [ ]:
display(pd.DataFrame({'Type': df.dtypes.astype(str), 'Distinct values': df.nunique(), 'Missing': df.isna().sum()}))

Each row is one campaign. The columns fall into three groups, and the split matters for everything that follows:

| Group | Columns | Use |
|---|---|---|
| Known **at launch** | `name`, `category`, `main_category`, `currency`, `country`, `goal`, `usd_goal_real`, `launched`, `deadline` | Possible model inputs |
| Known only **after** the campaign | `pledged`, `backers`, `usd pledged`, `usd_pledged_real`, `state` | `state` defines the target; the rest would be **leakage** and are never inputs |
| Identifiers | `ID` | Kept to join predictions back, never an input |

`usd_goal_real` is the goal converted to US dollars, so campaigns in different currencies can be compared; we use it instead of `goal`.

## 1.2 Campaign outcomes

In [ ]:
state_counts = df['state'].value_counts()
display(pd.DataFrame({'Campaigns': state_counts, 'Share': (state_counts / len(df)).round(3)}))

Only **successful** and **failed** campaigns have a clear, final outcome. **Canceled** campaigns were stopped by the creator (often to relaunch), **live** ones had not ended when the data was collected, and **undefined** and **suspended** ones have no reliable outcome. We train on successful versus failed only. About 40 % of all finished campaigns succeed; section 2 restricts the data to the campaigns an investor would actually consider.

## 1.3 Data quality

In [ ]:
launched = pd.to_datetime(df['launched'], errors='coerce')
deadline = pd.to_datetime(df['deadline'], errors='coerce')
print('Launch date before 2009 (Kickstarter opened in 2009; placeholder 1970):', (launched.dt.year < 2009).sum())
print("Country recorded as 'N,0\"' (broken value):", (df['country'] == 'N,0"').sum())
print('States of the broken-country rows:', df.loc[df['country'] == 'N,0"', 'state'].value_counts().to_dict())
print('Goal below 1 USD:', (df['usd_goal_real'] < 1).sum())
print('Missing names:', df['name'].isna().sum())
duration = (deadline - launched.dt.normalize()).dt.days
print('Campaign length (days), plausible launches only:',
      duration[launched.dt.year >= 2009].describe().round(1).to_dict())
print('Duplicate IDs:', df['ID'].duplicated().sum())

- **Launch dates in 1970** are a placeholder for "unknown" (7 campaigns). The campaign length and launch timing cannot be computed, so these become missing.
- **Country `N,0"`** is a broken value (3,797 campaigns), almost all with an *undefined* state; the country becomes missing.
- **Goals below 1 USD** are not real goals (38 campaigns); they become missing.
- **Missing names** (4) are left as empty titles.
- There are no duplicate campaigns.

All of these rules are fixed in advance and are applied the same way to new campaigns. Missing values are imputed inside the model, after the split.

## 1.4 What separates successful campaigns

In [ ]:
finished = df[df['state'].isin(['successful', 'failed'])].assign(
    success=lambda d: (d['state'] == 'successful').astype(int),
    log_goal=lambda d: np.log10(d['usd_goal_real'].clip(lower=1)),
    launch_year=lambda d: pd.to_datetime(d['launched'], errors='coerce').dt.year)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
cat_rate = finished.groupby('main_category')['success'].mean().sort_values()
axes[0, 0].barh(cat_rate.index, cat_rate.values, color='#356B9A')
axes[0, 0].axvline(finished['success'].mean(), linestyle='--', color='#B7791F')
axes[0, 0].set(title='Success rate by main category', xlabel='Success rate')
goal_bins = pd.cut(finished['usd_goal_real'], [0, 1e3, 5e3, 1e4, 5e4, 1e5, 1e6, np.inf],
                   labels=['<1k', '1-5k', '5-10k', '10-50k', '50-100k', '100k-1m', '>1m'])
goal_rate = finished.groupby(goal_bins, observed=True)['success'].mean()
axes[0, 1].bar(goal_rate.index.astype(str), goal_rate.values, color='#356B9A')
axes[0, 1].set(title='Success rate by goal (USD)', ylabel='Success rate')
year_rate = finished.groupby('launch_year')['success'].agg(['mean', 'size'])
axes[1, 0].plot(year_rate.index, year_rate['mean'], marker='o', color='#356B9A')
axes[1, 0].set(title='Success rate by launch year', ylabel='Success rate')
country_rate = finished.groupby('country')['success'].agg(['mean', 'size']).query('size >= 500')['mean'].sort_values()
axes[1, 1].barh(country_rate.index, country_rate.values, color='#356B9A')
axes[1, 1].set(title='Success rate by country (500+ campaigns)', xlabel='Success rate')
plt.tight_layout()
plt.show()

Read from the charts: the category matters a lot, and **smaller goals succeed far more often**, which is the strongest single pattern. The success rate also moves over the years, so a model trained on old campaigns may not transfer perfectly to new ones (we check this in section 5.4). Most campaigns come from the US.

# 2. Cleaning and validation rules

Fixed rules that learn nothing from the data, so they may run before the split.

**Business rule: the investor's universe.** A first run showed that the top of the shortlist was full of campaigns asking for 1 USD: they succeed easily but are not companies anyone would invest in. The investor only looks at campaigns with a goal of **at least 10,000 USD**, a serious amount for a product launch. In training we keep only those campaigns (about 123,000 finished ones, 28 % successful). New campaigns below the minimum are still scored by the pipeline (inference never drops rows), but they are outside the investor's universe and are filtered out of the shortlist. They become `clean_data.py` (training mode keeps only finished campaigns and builds the target; inference mode never drops rows) and `validate.py` (checks, never fixes).

In [ ]:
# PARAMETERS: cleaning and target (copy to TEAM_BRIEF.md / config.yaml)
ID_COLUMN = 'ID'
STATE_COLUMN = 'state'
TARGET = 'success'                                  # 1 = reached its goal
TRAINING_STATES = {'successful': 1, 'failed': 0}    # other states have no final outcome
FIRST_VALID_YEAR = 2009                             # Kickstarter opened in 2009
BROKEN_COUNTRY = 'N,0"'
MIN_GOAL_USD = 1.0
LEAKAGE_COLUMNS = ['pledged', 'backers', 'usd pledged', 'usd_pledged_real']   # only known after the campaign
MIN_INVESTABLE_GOAL_USD = 10_000                    # business rule: smaller campaigns are not investment targets

In [ ]:
def clean_campaigns(df_raw, training=True):
    """Stateless cleaning: the same input always gives the same output; nothing is learned from the data."""
    out = df_raw.drop(columns=LEAKAGE_COLUMNS + ['goal'], errors='ignore').copy()
    launched = pd.to_datetime(out['launched'], errors='coerce')
    deadline = pd.to_datetime(out['deadline'], errors='coerce')
    launched = launched.where(launched.dt.year >= FIRST_VALID_YEAR)
    out['goal_usd'] = out['usd_goal_real'].where(out['usd_goal_real'] >= MIN_GOAL_USD)
    out['duration_days'] = (deadline - launched.dt.normalize()).dt.days
    out['launch_month'] = launched.dt.month
    out['launch_year'] = launched.dt.year
    out['name'] = out['name'].fillna('')
    out['name_words'] = out['name'].str.split().str.len()
    out['country'] = out['country'].where(out['country'] != BROKEN_COUNTRY)
    out = out.drop(columns=['usd_goal_real', 'launched', 'deadline'])
    if training:   # rows are only dropped when we train; new campaigns are always scored
        out = out[out[STATE_COLUMN].isin(TRAINING_STATES)]
        out = out[out['goal_usd'] >= MIN_INVESTABLE_GOAL_USD].copy()   # the investor's universe
        out[TARGET] = out[STATE_COLUMN].map(TRAINING_STATES).astype(int)
    return out.reset_index(drop=True)


df_clean = clean_campaigns(pd.read_csv(RAW_DATA_PATH, low_memory=False))
print('Campaigns after cleaning:', len(df_clean),
      f'(removed {len(df) - len(df_clean)}: no final outcome, or goal below {MIN_INVESTABLE_GOAL_USD:,} USD)')
print('Success share:', round(df_clean[TARGET].mean(), 3))
print('Missing after cleaning:', df_clean.isna().sum()[lambda s: s > 0].to_dict())
df_clean.head()

In [ ]:
# Validation rules (become validate.py): the pipeline stops with a clear error if any fails
REQUIRED_COLUMNS = [ID_COLUMN, 'name', 'category', 'main_category', 'country', 'goal_usd',
                    'duration_days', 'launch_month', 'launch_year', 'name_words']
DURATION_RANGE = (1, 92)              # Kickstarter campaigns last at most about 90 days
LAUNCH_YEAR_RANGE = (2009, 2030)


def check_rules(data):
    problems = [c for c in REQUIRED_COLUMNS if c not in data]
    if data[ID_COLUMN].duplicated().any():
        problems.append('duplicate IDs')
    if not data['duration_days'].dropna().between(*DURATION_RANGE).all():
        problems.append('duration out of range')
    if not data['launch_year'].dropna().between(*LAUNCH_YEAR_RANGE).all():
        problems.append('launch year out of range')
    if (data['goal_usd'].dropna() <= 0).any():
        problems.append('non-positive goal')
    if set(LEAKAGE_COLUMNS) & set(data.columns):
        problems.append('leakage columns present')
    if TARGET in data and not data[TARGET].isin([0, 1]).all():
        problems.append('target not binary')
    return problems


print('Validation problems:', check_rules(df_clean) or 'none')

# 3. Split before anything is fitted

The pipeline splits **once, in `main.py`**, into train (60 %), validation (20 %) and test (20 %), stratified on the target. Train is for fitting and tuning, validation for comparing models, the quality gate and the probability cutoff, and test is opened **once** at the very end.

In [ ]:
# PARAMETERS: inputs and split (copy to TEAM_BRIEF.md / config.yaml)
NUMERIC_FEATURES = ['goal_usd', 'duration_days', 'launch_month', 'launch_year', 'name_words']
LOG_FEATURES = ['goal_usd']                                   # log-transformed: goals span 1 USD to 166 m USD
CATEGORICAL_FEATURES = ['main_category', 'category', 'country']
EXCLUDED = {'ID': 'identifier', 'name': 'raw text; only its word count is used', 'currency': 'duplicates country',
            'state': 'defines the target', 'pledged, backers, usd pledged, usd_pledged_real': 'leakage'}
TEST_SIZE = 0.20
VAL_SIZE = 0.20
RANDOM_STATE = 42

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X, y = df_clean[FEATURES], df_clean[TARGET]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=VAL_SIZE / (1 - TEST_SIZE),
                                                  stratify=y_rest, random_state=RANDOM_STATE)
for name, part in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    print(f'{name:<10} {len(part):>7} campaigns, success share {part.mean():.3f}')

# 4. Model comparison

## 4.1 Success measure, fixed before any results

- **ML metric for the quality gate:** `roc_auc` on the validation split, **higher is better**, **at least 0.70**. ROC AUC measures how well the model ranks campaigns from most to least likely to succeed, which is exactly what a shortlist needs.
- **Business metric:** the **hit rate of the shortlist**, meaning the share of successful campaigns among the 10 % the model ranks highest (`precision_top_10pct`), compared with the hit rate an investor gets by picking at random (printed below).
- **Non-AI baseline:** a rule an investor could apply without a model, "prefer small goals", scored with the same metric.
- **Model choice rule, also fixed now:** we take the model with the best validation ROC AUC, **unless the logistic regression is within 0.01 of it**, in which case we take the logistic regression because it is easier to explain and audit.

In [ ]:
# PARAMETERS: metric, quality gate and model-choice rule (must match README.md, TEAM_BRIEF.md and config.yaml)
METRIC = 'roc_auc'
GATE_THRESHOLD = 0.70
GATE_DIRECTION = 'higher_is_better'
SHORTLIST_SHARE = 0.10
PREFER_LOGIT_WITHIN = 0.01

In [ ]:
from sklearn.metrics import roc_auc_score


def precision_top(y_true, scores, share=SHORTLIST_SHARE):
    """Share of successes among the top `share` of campaigns ranked by score."""
    n = max(1, int(len(scores) * share))
    top = np.argsort(-np.asarray(scores))[:n]
    return float(np.asarray(y_true)[top].mean())


baseline_scores = -X_val['goal_usd'].fillna(X_train['goal_usd'].median())   # smaller goal = more promising
print(f'Random pick hit rate: {y_val.mean():.3f}')
print(f'Baseline rule (small goals first): ROC AUC {roc_auc_score(y_val, baseline_scores):.3f}, '
      f'top-10% hit rate {precision_top(y_val, baseline_scores):.3f}')

## 4.2 One recipe, three models

The feature recipe (this is what `features.py` returns, **unfitted**): numeric inputs get a median imputer with a missing-value indicator, the goal is log-transformed (log1p, so the 0/1 missing flag stays 0/1-like), everything numeric is scaled; categories get an "Unknown" imputer and one-hot encoding with rare categories grouped. Each model is tuned with 5-fold cross-validation on a random sample of the training rows (to keep tuning to a few minutes), then refitted on all training rows.

In [ ]:
# PARAMETERS: tuning (notebook only; the pipeline receives the final values)
RARE_CATEGORY_MINIMUM = 100
CV_FOLDS = 5
TUNE_SAMPLE = 60_000
CANDIDATES = {
    'logistic_regression': {'model__C': [0.01, 0.03, 0.1, 0.3, 1.0]},
    'decision_tree': {'model__max_depth': [4, 6, 8, 10, 12]},
    'random_forest': {'model__max_depth': [8, 14, 20]},
}

In [ ]:
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV

# Keep the class syntax (penalty='l1') across scikit-learn versions; newer versions only warn about it
warnings.filterwarnings('ignore', message=".*'penalty' was deprecated.*", category=FutureWarning)
warnings.filterwarnings('ignore', message='Inconsistent values: penalty=.*', category=UserWarning)


def make_recipe():
    """Unfitted recipe: this is what features.py returns."""
    other_numeric = [c for c in NUMERIC_FEATURES if c not in LOG_FEATURES]
    log_numeric = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                            ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
                            ('scale', StandardScaler())])
    numeric = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                        ('scale', StandardScaler())])
    categorical = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
                            ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                     min_frequency=RARE_CATEGORY_MINIMUM))])
    return ColumnTransformer([('log', log_numeric, LOG_FEATURES), ('num', numeric, other_numeric),
                              ('cat', categorical, CATEGORICAL_FEATURES)], verbose_feature_names_out=False)


MODELS = {
    'logistic_regression': LogisticRegression(penalty='l1', solver='liblinear', max_iter=2000,
                                              random_state=RANDOM_STATE),
    'decision_tree': DecisionTreeClassifier(min_samples_leaf=50, random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(n_estimators=150, min_samples_leaf=20, n_jobs=-1,
                                            random_state=RANDOM_STATE),
}

tune_idx = X_train.sample(n=min(TUNE_SAMPLE, len(X_train)), random_state=RANDOM_STATE).index
cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
results, fitted = [], {}
for name, estimator in MODELS.items():
    grid = GridSearchCV(Pipeline([('preprocess', make_recipe()), ('model', estimator)]), CANDIDATES[name],
                        scoring=METRIC, cv=cv, return_train_score=True, n_jobs=1)
    grid.fit(X_train.loc[tune_idx], y_train.loc[tune_idx])
    model = grid.best_estimator_.fit(X_train, y_train)          # refit on all training rows
    fitted[name] = model
    val_scores = model.predict_proba(X_val)[:, 1]
    best = grid.cv_results_['mean_test_score'][grid.best_index_]
    results.append({'model': name, 'best params': grid.best_params_,
                    'CV train ROC AUC': grid.cv_results_['mean_train_score'][grid.best_index_],
                    'CV ROC AUC': best, 'validation ROC AUC': roc_auc_score(y_val, val_scores),
                    'validation top-10% hit rate': precision_top(y_val, val_scores)})
comparison = pd.DataFrame(results).set_index('model')
display(comparison.round(4))

In [ ]:
best_name = comparison['validation ROC AUC'].idxmax()
logit_gap = comparison.loc[best_name, 'validation ROC AUC'] - comparison.loc['logistic_regression', 'validation ROC AUC']
CHOSEN = 'logistic_regression' if logit_gap <= PREFER_LOGIT_WITHIN else best_name
CHOSEN_PARAMS = {k.replace('model__', ''): v for k, v in comparison.loc[CHOSEN, 'best params'].items()}
best_model = fitted[CHOSEN]
print(f'Best on validation: {best_name} | logit gap: {logit_gap:.4f} | chosen: {CHOSEN} {CHOSEN_PARAMS}')

**Reading the comparison (this run):** the random forest ranks best (validation ROC AUC 0.737), ahead of the logistic regression (0.717) and the decision tree (0.711). The tree has the largest train-CV gap, so it overfits most. The logit is 0.020 behind the forest, more than the 0.01 margin fixed in 4.1, so by our own rule we **choose the random forest** (150 trees, max depth 20, at least 20 campaigns per leaf). We lose the line-by-line explanation of a logit, so section 5.2 shows which inputs the forest relies on most.

# 5. The chosen model

## 5.1 Quality gate and probability cutoff (validation)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve,
                             RocCurveDisplay)

val_proba = best_model.predict_proba(X_val)[:, 1]
val_score = roc_auc_score(y_val, val_proba)
print(f'Validation {METRIC} = {val_score:.4f} vs gate {GATE_THRESHOLD} -> '
      f'{"MET" if val_score >= GATE_THRESHOLD else "NOT MET"}')

# Cutoff that maximises F1 on validation (turns probabilities into yes/no)
precision, recall, cutoffs = precision_recall_curve(y_val, val_proba)
f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
CUTOFF = float(round(cutoffs[np.argmax(f1)], 2))
val_pred = (val_proba >= CUTOFF).astype(int)
display(pd.Series({'cutoff': CUTOFF, 'accuracy': accuracy_score(y_val, val_pred),
                   'precision': precision_score(y_val, val_pred), 'recall': recall_score(y_val, val_pred),
                   'f1': f1_score(y_val, val_pred), 'top-10% hit rate': precision_top(y_val, val_proba),
                   'random-pick hit rate': y_val.mean()}).round(3))
ax = plt.gca()
for name, model in fitted.items():
    RocCurveDisplay.from_predictions(y_val, model.predict_proba(X_val)[:, 1], name=name, ax=ax)
ax.plot([0, 1], [0, 1], '--', color='#777777')
plt.show()

## 5.2 Drivers of success

In [ ]:
names = best_model.named_steps['preprocess'].get_feature_names_out()
step = best_model.named_steps['model']
if hasattr(step, 'coef_'):
    effect = pd.Series(step.coef_[0], index=names)
    print(f'{(effect != 0).sum()} of {len(effect)} inputs kept by the Lasso')
    top = pd.concat([effect.sort_values().head(8), effect.sort_values().tail(8)])
    display(pd.DataFrame({'Coefficient': top, 'Odds ratio': np.exp(top)}).round(2))
else:
    effect = pd.Series(step.feature_importances_, index=names).sort_values(ascending=False)
    display(effect.head(15).round(3).rename('Importance'))

For the logit, numeric inputs are standardised (the goal on a log scale), so odds ratios are **per standard deviation**; category effects are relative to the other categories and every category is kept so that new categories are handled safely. Read them as directions and relative strength, not causes.

## 5.3 Stability over time

Our split is random, but an investor always predicts **future** campaigns. As a check, we score the validation campaigns by launch year: if the ROC AUC drops sharply for recent years, the model needs regular retraining (a monitoring point for the business case).

In [ ]:
by_year = pd.DataFrame({'year': X_val['launch_year'], 'y': y_val, 'p': val_proba}).dropna()
display(by_year.groupby('year').apply(lambda g: pd.Series({'campaigns': len(g), 'success rate': g['y'].mean(),
                                                           'ROC AUC': roc_auc_score(g['y'], g['p'])}),
                                      include_groups=False).round(3))

## 5.4 Test split: opened once

In [ ]:
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= CUTOFF).astype(int)
display(pd.Series({'roc_auc': roc_auc_score(y_test, test_proba), 'accuracy': accuracy_score(y_test, test_pred),
                   'precision': precision_score(y_test, test_pred), 'recall': recall_score(y_test, test_pred),
                   'top-10% hit rate': precision_top(y_test, test_proba),
                   'random-pick hit rate': y_test.mean()}).round(3))

If the test numbers are close to validation, the model generalises and the gate was not luck. We do not go back and tune after seeing them.

# 6. Business use: the investor's shortlist

Each week the investor looks at the campaigns that just launched. We simulate this on the validation campaigns (the model never trained on them): rank them by predicted probability and keep the top 10 %.

In [ ]:
val_view = df_clean.loc[X_val.index, ['name', 'main_category', 'category', 'country', 'goal_usd',
                                      'duration_days', TARGET]].assign(probability=val_proba)
shortlist = val_view.nlargest(int(len(val_view) * SHORTLIST_SHARE), 'probability')
print(f'Shortlist: {len(shortlist)} of {len(val_view)} campaigns')
print(f'Hit rate: shortlist {shortlist[TARGET].mean():.1%} vs random pick {val_view[TARGET].mean():.1%} '
      f'(x{shortlist[TARGET].mean() / val_view[TARGET].mean():.2f})')
display(shortlist['main_category'].value_counts().head(8).rename('campaigns on the shortlist'))
display(shortlist.head(10).round(3))

# 7. Final choices for the pipeline

Copy these values into `TEAM_BRIEF.md`, the `SETTINGS` in `main.py` and `config.yaml`. The metric, threshold and direction must be **identical** in `README.md`, `TEAM_BRIEF.md` and `config.yaml`. This notebook stays frozen in `notebooks/` once the team agrees.

In [ ]:
import json

final_choices = {
    'data': {'raw_file': 'data/raw/ks-projects-201801.csv', 'id_column': ID_COLUMN, 'state_column': STATE_COLUMN,
             'target': TARGET, 'target_rule': 'state == successful (1) vs failed (0)',
             'problem_type': 'classification'},
    'cleaning': {'training_states': TRAINING_STATES, 'min_investable_goal_usd': MIN_INVESTABLE_GOAL_USD, 'first_valid_year': FIRST_VALID_YEAR,
                 'broken_country': BROKEN_COUNTRY, 'min_goal_usd': MIN_GOAL_USD, 'leakage_columns': LEAKAGE_COLUMNS,
                 'derived': ['goal_usd', 'duration_days', 'launch_month', 'launch_year', 'name_words']},
    'validation': {'required_columns': REQUIRED_COLUMNS, 'duration_range': list(DURATION_RANGE),
                   'launch_year_range': list(LAUNCH_YEAR_RANGE)},
    'features': {'numeric': NUMERIC_FEATURES, 'log': LOG_FEATURES, 'categorical': CATEGORICAL_FEATURES,
                 'rare_category_minimum': RARE_CATEGORY_MINIMUM, 'excluded': EXCLUDED},
    'split': {'test_size': TEST_SIZE, 'val_size': VAL_SIZE, 'random_state': RANDOM_STATE, 'stratify': True},
    'model': {'type': CHOSEN, 'params': {k: (v if not isinstance(v, np.generic) else v.item())
                                         for k, v in CHOSEN_PARAMS.items()},
              'alternatives_compared': list(MODELS), 'random_state': RANDOM_STATE},
    'evaluation': {'metric': METRIC, 'threshold': GATE_THRESHOLD, 'direction': GATE_DIRECTION,
                   'probability_cutoff': CUTOFF, 'business_metric': 'precision_top_10pct'},
}
print(json.dumps(final_choices, indent=2, default=str))

## Summary

- **Data:** 378,661 Kickstarter campaigns. The investor's universe is finished campaigns (successful or failed) with a goal of at least 10,000 USD: 122,881 campaigns, 28.4 % successful.
- **Inputs:** only what is visible at launch: category, main category, country, goal in USD, campaign length, launch month and year, number of words in the title. Pledged amounts and backers are leakage and never used.
- **Models compared (validation ROC AUC):** random forest 0.737, logistic regression 0.717, decision tree 0.711. Random forest chosen by the rule fixed before results.
- **Quality gate:** validation ROC AUC 0.737 against at least 0.70: **met**. Test ROC AUC 0.743, so the result holds.
- **Business use:** the weekly top-10 % shortlist succeeds 58.6 % of the time, against 28.4 % for a random pick (x2.1) and 37.2 % for the "small goals first" rule. The shortlist is led by board and card games, music and design projects.
- **Most important inputs:** goal size, title length, campaign length, and whether it is a tabletop game.
- **Stability:** ROC AUC by launch year rises from about 0.62-0.68 (2010-2013) to about 0.75-0.77 (2014-2017); success rates move between 23 % and 35 %, so the model needs monitoring and retraining. (2009 has only 38 campaigns in validation, too few to read.)
- **Limits:** funding success is a proxy for demand, not investment returns; campaigns from 2009 to early 2018 only; the 10,000 USD minimum is a business choice.